## Setup

In [ ]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import sys
import os
from IPython.display import display, HTML

sys.path.insert(0, os.path.abspath('../pytorch-physics/'))
sys.path.insert(0, os.path.abspath('../pytorch-geometric/'))
sys.path.insert(0, os.path.abspath('../utils/'))
from boid_tracking import *
from boid import Flock
from coordinate_orientaitons import *
from helper import html

In [ ]:
# temporary cool, but then one unit
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N = 2000
box_top = 100

flock_args = {
    'D': 2,
    'N': N,
    'box_top': box_top,
    'pass_through_edges': True,
    'bouncy_edges': False,
    'device': device,
}

boid_args = {
    'init_speed': None,
    # 'min_speed': 3,
    # 'max_speed': 6,
    # 'max_acc': 0.5,

    'min_speed': 3/9,
    'max_speed': 6/9,
    'max_acc': 0.5/9,
    
    'view_radius': 10,
    'view_angle': None,
    
    'avoid_radius': 8,
    'avoid_view': True,
    
    'sep_factor': 0.5,    # avoidfactor
    'align_factor': 0.05,  # matchingfactor
    'cohe_factor': 0.005,  # centeringfactor
    'bias_factor': 0.005,
    'edge_factor': 0.05,
    
    'is_debug': False
}

### HTML

In [ ]:
model_html = """
<!DOCTYPE html> <html><!--
 Page saved with SingleFile 
 url: http://localhost:3001/html/06-boid-id-tracking.html 
 saved date: Thu Oct 31 2024 20:02:38 GMT+0100 (Central European Standard Time)
--><meta charset=utf-8>
<meta name=viewport content="width=device-width initial-scale=1">
<style>html{overflow-x:initial!important}:root{--bg-color:#ffffff;--text-color:#333333;--select-text-bg-color:#B5D6FC;--select-text-font-color:auto;--monospace:"Lucida Console",Consolas,"Courier",monospace;--title-bar-height:20px}html{font-size:14px;background-color:var(--bg-color);color:var(--text-color);font-family:"Helvetica Neue",Helvetica,Arial,sans-serif;-webkit-font-smoothing:antialiased}body{margin:0px;padding:0px;height:auto;inset:0px;font-size:1rem;line-height:1.428571;overflow-x:hidden;background:inherit}.in-text-selection,::selection{text-shadow:none;background:var(--select-text-bg-color);color:var(--select-text-font-color)}#write{height:auto;width:inherit;word-break:normal;overflow-wrap:break-word;position:relative;white-space:normal;overflow-x:visible;padding-top:36px}body.typora-export{padding-left:30px;padding-right:30px}.typora-export p{white-space:pre-wrap}@media screen and (max-width:500px){body.typora-export{padding-left:0px;padding-right:0px}#write{padding-left:20px;padding-right:20px}}img{vertical-align:middle;image-orientation:from-image}*,::after,::before{box-sizing:border-box}#write p{width:inherit}#write p{position:relative}p{line-height:inherit}p{orphans:4}p>.md-image:only-child:not(.md-img-error) img,p>img:only-child{display:block;margin:auto}@media screen and (max-width:48em){}@media screen and (max-width:1024px){}@media screen and (max-width:800px){}:root{--text-color:#1f2329;--bg-color:white;--side-bar-bg-color:white;--active-file-bg-color:white;--rawblock-edit-panel-bd:#f5f6f7;--window-border:0 solid white;--control-text-color:#1f2329;--primary-color:#3370ff;--primary-btn-border-color:#3370ff;--active-file-border-color:#3370ff;--primary-btn-text-color:#3370ff;--item-hover-text-color:#3370ff;--meta-content-color:#3370ff;--search-select-text-color:#3370ff;--heading-char-color:#3370ff;--mermaid-theme:default}#write{line-height:1.68;-webkit-text-size-adjust:100%;max-width:960px;margin:0px auto;padding:1.6rem 3.2rem;font-family:-apple-system,BlinkMacSystemFont,Helvetica Neue,Arial,Segoe UI,PingFang SC,Microsoft Yahei,Hiragino Sans GB,sans-serif,Apple Color Emoji,Segoe UI Emoji,Segoe UI Symbol,Noto Color Emoji;font-size:16px;color:#1f2329}#write *{-moz-box-sizing:border-box;-webkit-box-sizing:border-box;box-sizing:border-box}#write>*:first-child{margin-top:0!important}#write>*:last-child{margin-top:0!important}#write p{margin-top:8px;margin-bottom:8px}#write p{margin-left:0;margin-right:0}#write img{max-width:100%;height:auto}#write *{-webkit-font-smoothing:antialiased;text-rendering:optimizeLegibility}</style><title>06-boid-id-tracking</title>
<meta name=referrer content=no-referrer><link rel=canonical href=http://localhost:3001/html/06-boid-id-tracking.html><meta http-equiv=content-security-policy content="default-src 'none'; font-src 'self' data:; img-src 'self' data:; style-src 'unsafe-inline'; media-src 'self' data:; script-src 'unsafe-inline' data:; object-src 'self' data:; frame-src 'self' data:;"><style>img[src="data:,"],source[src="data:,"]{display:none!important}</style></head>
<body class=typora-export><div class=typora-export-content>
<div id=write><p><img src=... style=width:85%><p><img src=... style=width:85%><p><img src="..." style=width:83%><p>&nbsp;</p></div></div>
"""

## Visualization

In [ ]:
flock = Flock(
    **flock_args,
    **boid_args
)

In [ ]:
list_of_birds_pos = []
list_of_birds_vel = []

iterations = 50

for _ in range(iterations):
    flock.update()
    list_of_birds_pos.append(flock.pos)
    list_of_birds_vel.append(flock.vel)

vid_len = 50
visualize_boids(list_of_birds_pos[iterations-vid_len:], list_of_birds_vel[iterations-vid_len:], [0, box_top])

In [ ]:
def update_and_calc_flocks(flock, n=5, tracked_indicies=np.array([])):
    flock.update()
    flock_pos = flock.pos.cpu().numpy()
    flock_vel = flock.vel.cpu().numpy()

    if tracked_indicies.shape[0] == 0:
        # choose  one boid from the area [40, 60]^2
        window_flocks = (flock_pos[:, 0] >= 40) & (flock_pos[:, 0] <= 60) & (flock_pos[:, 1] >= 40) & (flock_pos[:, 1] <= 60)
        window_flocks_abs_indicies = np.arange(0, flock_pos.shape[0])[window_flocks]
        tracked_flock_ind = np.random.choice(window_flocks_abs_indicies)
        tracked_flock_pos = flock_pos[tracked_flock_ind]
        # find the n*2-closests boids
        n_nearest_indices = np.argsort(np.linalg.norm(flock_pos - tracked_flock_pos, axis=1))[:(2*n)]
        # choose n unique boids at random
        tracked_indicies = np.random.choice(n_nearest_indices, size=n, replace=False)

    flocks_with_ids = np.column_stack([flock_pos[tracked_indicies], flock_vel[tracked_indicies], tracked_indicies])
    
    return flocks_with_ids

In [ ]:
def calc_inidcies(flock, n=5, tracked_indicies=np.array([])):
    flock.update()
    flock_pos = flock.pos.cpu().numpy()
    flock_vel = flock.vel.cpu().numpy()

    # choose  one boid from the area [40, 60]^2
    window_flocks = (flock_pos[:, 0] >= 40) & (flock_pos[:, 0] <= 60) & (flock_pos[:, 1] >= 40) & (flock_pos[:, 1] <= 60)
    window_flocks_abs_indicies = np.arange(0, flock_pos.shape[0])[window_flocks]
    tracked_flock_ind = np.random.choice(window_flocks_abs_indicies)
    tracked_flock_pos = flock_pos[tracked_flock_ind]
    # find the n*2-closests boids
    n_nearest_indices = np.argsort(np.linalg.norm(flock_pos - tracked_flock_pos, axis=1))[:(2*n)]
    # choose n unique boids at random
    tracked_indicies = np.random.choice(n_nearest_indices, size=n, replace=False)

    return tracked_indicies

In [ ]:
import matplotlib.colors as mcolors

def visualize_boids_multiple_timeframes(birds_pos, birds_vel, birds_id, boundaries):
    birds_pos = birds_pos.cpu().numpy()
    birds_vel = birds_vel.cpu().numpy()
    birds_id = birds_id.cpu().numpy()
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(boundaries[0], boundaries[1])
    ax.set_ylim(boundaries[0], boundaries[1])
    ax.set_xlabel('X-axis')
    ax.set_ylabel('Y-axis')
    
    initial_positions = birds_pos
    initial_velocities = birds_vel
    
    num_birds = len(np.unique(birds_id))
    cmap = plt.cm.get_cmap('rainbow')
    
    norm = mcolors.Normalize(vmin=birds_id.min(), vmax=birds_id.max())
    colors = cmap(norm(birds_id))
    
    scatter = ax.scatter(initial_positions[:, 0], initial_positions[:, 1], c=colors, s=10)
    
    quiver = ax.quiver(initial_positions[:, 0], initial_positions[:, 1], 
                       initial_velocities[:, 0], initial_velocities[:, 1], 
                       color=colors, angles='xy', scale_units='xy', scale=1)
    
    plt.show()
    plt.close(fig)

In [ ]:
n, time_steps, iterations = 5, 10, 20

tracked_indicies = np.array([])
list_of_birds_pos = []
list_of_birds_vel = []
list_of_birds_ind = []

for i in range(iterations):
    if i % time_steps == 0:
        flocks = update_and_calc_flocks(flock, n)
        tracked_indicies = flocks[:, 4].astype(int)
    else:
        flocks = update_and_calc_flocks(flock, n, tracked_indicies)

    list_of_birds_pos.append(torch.tensor(flocks[:, 0:2]))
    list_of_birds_vel.append(torch.tensor(flocks[:, 2:4]))    
    list_of_birds_ind.append(torch.tensor(tracked_indicies))

In [ ]:
offset = 10
torch_bird_pos = torch.stack(list_of_birds_pos[offset+0:offset+time_steps]).reshape(-1, 2)
torch_bird_vel = torch.stack(list_of_birds_vel[offset+0:offset+time_steps]).reshape(-1, 2)
torch_bird_ind = torch.stack(list_of_birds_ind[offset+0:offset+time_steps]).reshape(-1, 1)

visualize_boids_multiple_timeframes(torch_bird_pos, torch_bird_vel, torch_bird_ind, [30, 70])

## Network-Idea

In [ ]:
HTML(model_html)

## Graph - Postion Prediction Model

In [ ]:
n, time_steps, iterations = 5, 2, 2

def calc_trainings_data():
    birds_pos = np.zeros((time_steps, n, 2))
    birds_vel = np.zeros((time_steps, n, 2))
    birds_ind = np.zeros((time_steps, n), dtype=int)
    
    flock.update()
    flock_pos = flock.pos.cpu().numpy()
    flock_vel = flock.vel.cpu().numpy()
    
    window_flocks = (flock_pos[:, 0] >= 40) & (flock_pos[:, 0] <= 60) & (flock_pos[:, 1] >= 40) & (flock_pos[:, 1] <= 60)
    window_flocks_abs_indicies = np.arange(0, flock_pos.shape[0])[window_flocks]
    tracked_flock_ind = np.random.choice(window_flocks_abs_indicies)
    tracked_flock_pos = flock_pos[tracked_flock_ind]
    n_nearest_indices = np.argsort(np.linalg.norm(flock_pos - tracked_flock_pos, axis=1))[:(2*n)]
    tracked_indices = np.random.choice(n_nearest_indices, size=n, replace=False)

    birds_pos[0] = flock_pos[tracked_indices]
    birds_vel[0] = flock_vel[tracked_indices]
    birds_ind[0] = tracked_indices
    
    flock.update()
    flock_pos = flock.pos.cpu().numpy()
    flock_vel = flock.vel.cpu().numpy()
    birds_pos[1] = flock_pos[tracked_indices]
    birds_vel[1] = flock_vel[tracked_indices]
    birds_ind[1] = tracked_indices

    flocks_pos = torch.from_numpy(birds_pos.reshape(-1, 2))
    flocks_vel = torch.from_numpy(birds_vel.reshape(-1, 2))
    flocks_id = torch.from_numpy(birds_ind.reshape(-1, 1))
    
    return create_id_tracking_graph_gnn_training(flocks_pos, flocks_vel, flocks_id)

In [ ]:
def create_id_tracking_graph_gnn_training(flocks_pos, flocks_vel, flocks_id):
    unique_ids = torch.unique(flocks_id)
    # two time-steps mean two iterations of flocks
    time_steps = torch.floor_divide(flocks_id.shape[0], unique_ids.shape[0]).item()
    node_features = torch.arange(0, flocks_pos.shape[0])
    
    edge_connections = np.array(np.meshgrid(np.arange(0, unique_ids.shape[0]), np.arange(0, flocks_pos.shape[0])))
    edge_connections = edge_connections.T.reshape(-1, 2)
    edge_connections = edge_connections[edge_connections[:, 0] != edge_connections[:, 1]].astype(int)

    # filter out edges to suit the trainings-graph
    node_count = unique_ids.max() + 1
    shuffle = torch.randperm(node_count)
    tracked_ids = shuffle[1:4]
    to_predict_id = shuffle[0]
    edge_filter = (data.edge_index[0, :] == node_val) & (data.edge_index[1, :] != (to_predict_id + num_nodes))
    
    dists, angles = dist_angle_from_matrix(np.concatenate([flocks_pos, flocks_pos]), edge_connections)
    vel_rad = velocity_vector_rad(np.vstack([flocks_vel, flocks_vel]), edge_connections)
    rad_angles = np.deg2rad(angles)
    # there is one less edge than number of nodes
    time_diff = (torch.arange(0, time_steps).repeat_interleave(unique_ids.shape[0])[1:]).repeat(unique_ids.shape[0])

    assert(time_diff.shape[0] == edge_connections.shape[0])
    
    dists = torch.tensor(dists, dtype=torch.float32)
    rad_angles = torch.tensor(rad_angles, dtype=torch.float32)
    vel_rad = torch.tensor(vel_rad, dtype=torch.float32)
    time_diff = torch.tensor(time_diff, dtype=torch.float32)

    edge_features = torch.column_stack([dists, rad_angles, vel_rad, time_diff])
    edge_connections = torch.tensor(edge_connections)
    shuffle = torch.randperm(edge_features.shape[0])
    edge_features = edge_features[shuffle]
    edge_connections = edge_connections[shuffle]

    data = Data(
        x=node_features,
        edge_index=edge_connections.t().contiguous(),
        edge_attr=edge_features
    )

    return data

In [ ]:
data = calc_trainings_data()

In [ ]:
node_count = unique_ids.max() + 1
shuffle = torch.randperm(node_count)
tracked_ids = shuffle[1:4]
to_predict_id = shuffle[0]
filter = (data.edge_index[0, :] == node_val) & (data.edge_index[1, :] != (to_predict_id + num_nodes))
data.edge_attr[filter]
# data.edge_index, to_predict_id

In [ ]:
filter

In [ ]:
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader

# Hard coded for 3-boids with 5 timeframes
class EdgeAttrPredictor(nn.Module):
    # in_channels: number of edge-features for the node: (3*5-1)*4 = 56
    # out_channels: predicting 1 value for each edge:  3*5-1=14
    def __init__(self, in_channels=56, out_channels=14):
        super(EdgeAttrPredictor, self).__init__()
        
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, out_channels),
            nn.Softmax(dim=-1)
        )
        self.mlp.apply(self.weights_init)
    
    def forward(self, data):
        node_count = data.edge_index[0].max() + 1
        shuffle = torch.randperm(node_count)
        tracked_ids = shuffle[:3]
        to_predict_id = shuffle[3]
        
        edge_outputs = []
        for node_val in tracked_ids:
            filter_edges = (data.edge_index[0, :] == node_val) & (data.edge_index[1, :] != (to_predict_id + node_count))
            rel = data.edge_attr[filter_edges]
            x = self.mlp(torch.flatten(rel))
            edge_outputs.append(x)

        return torch.vstack(edge_outputs)

    @staticmethod
    def weights_init(m):
        if isinstance(m, nn.Linear):
            init.kaiming_normal_(m.weight, nonlinearity='relu')

    @staticmethod
    def predict_labels():
        pass

In [ ]:
def calc_simple_loss(data, edge_val):
    n = 3
    time_steps = 5
    mod_edge = torch.remainder(data.edge_index[1] - data.edge_index[0],  n)
    is_same_obj = (mod_edge == torch.zeros(mod_edge.shape[0])).float() / (time_steps - 1)

    return F.mse_loss(is_same_obj, edge_val)

In [ ]:
n, time_steps, iterations = 3, 5, 5

tracked_indicies = np.array([])
list_of_birds_pos = []
list_of_birds_vel = []
list_of_birds_ind = []

for i in range(iterations):
    if i % time_steps == 0:
        flocks = update_and_calc_flocks(flock, n)
        tracked_indicies = flocks[:, 4].astype(int)
    else:
        flocks = update_and_calc_flocks(flock, n, tracked_indicies)

    list_of_birds_pos.append(torch.tensor(flocks[:, 0:2]))
    list_of_birds_vel.append(torch.tensor(flocks[:, 2:4]))    
    list_of_birds_ind.append(torch.tensor(tracked_indicies))

In [ ]:
offset = 0
flocks_pos = torch.stack(list_of_birds_pos[offset+0:offset+time_steps]).reshape(-1, 2)
flocks_vel = torch.stack(list_of_birds_vel[offset+0:offset+time_steps]).reshape(-1, 2)
flocks_id = torch.stack(list_of_birds_ind[offset+0:offset+time_steps]).reshape(-1, 1)

data = create_id_tracking_graph_gnn_training(flocks_pos, flocks_vel, flocks_id)

In [ ]:
def calc_inidcies(flock_pos, flock_vel, n=3):
    flock.update()
    flock_pos = flock.pos.cpu().numpy()
    flock_vel = flock.vel.cpu().numpy()

    # choose  one boid from the area [40, 60]^2
    window_flocks = (flock_pos[:, 0] >= 40) & (flock_pos[:, 0] <= 60) & (flock_pos[:, 1] >= 40) & (flock_pos[:, 1] <= 60)
    window_flocks_abs_indicies = np.arange(0, flock_pos.shape[0])[window_flocks]
    tracked_flock_ind = np.random.choice(window_flocks_abs_indicies)
    tracked_flock_pos = flock_pos[tracked_flock_ind]
    # find the n*2-closests boids
    n_nearest_indices = np.argsort(np.linalg.norm(flock_pos - tracked_flock_pos, axis=1))[:(2*n)]
    # choose n unique boids at random
    tracked_indicies = np.random.choice(n_nearest_indices, size=n, replace=False)

    return tracked_indicies

In [ ]:
model = EdgeAttrPredictor()

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)

def train():
    model.train()
    optimizer.zero_grad()
    
    data = calc_trainings_data()
    predicted_edge_attr = torch.flatten(model(data))
    loss = calc_simple_loss(data, predicted_edge_attr)

    loss.backward()
    optimizer.step()
    
    return loss.item()

num_epochs = 5000
total_loss = 0
count = 0

for epoch in range(num_epochs):
    loss = train()
    total_loss += loss
    count += 1
    
    if (epoch + 1) % 1000 == 0:
        average_loss = total_loss / count
        print(f'Epoch {epoch+1}/{num_epochs}, Average Loss: {average_loss:.4f}')
        total_loss = 0
        count = 0

In [ ]:
data = calc_trainings_data()
predicted_edge_attr = torch.flatten(model(data))
loss = calc_simple_loss(data, predicted_edge_attr)

In [ ]:
predicted_edge_attr